# 1. Source audit and reproducible cleaning

One row represents a restaurant listing. Raw inputs are preserved. The pipeline validates keys, flags missing cuisines, joins countries and creates a separate cuisine bridge. Zero ratings are treated as unrated, not poor reviews.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd


In [2]:
raw = pd.read_csv(ROOT/'data/raw/zomato.csv', encoding='latin1')
countries = pd.read_excel(ROOT/'data/raw/Country-Code.xlsx')
print('Source shape:', raw.shape)
display(raw.head(3))
display(raw.isna().sum().to_frame('missing'))

Source shape: (9551, 21)


,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270


,missing
Restaurant ID,0
Restaurant Name,0
Country Code,0
City,0
Address,0
Locality,0
Locality Verbose,0
Longitude,0
Latitude,0
Cuisines,9


In [3]:
from src.pipeline import clean_data, run_pipeline
clean, cuisines, audit = clean_data(raw, countries)
display(pd.Series(audit).to_frame('value'))
display(clean[['restaurant_id','country','aggregate_rating','rating_for_analysis','cuisines_missing']].head())
assert clean.restaurant_id.is_unique
assert len(clean) == len(raw.drop_duplicates())
assert clean.loc[~clean.is_rated,'rating_for_analysis'].isna().all()

,value
raw_rows,9551
clean_rows,9551
exact_duplicates_removed,0
duplicate_restaurant_ids,0
missing_cuisines,9
unrated_restaurants,2148
zero_cost_records,18
zero_coordinate_pairs,497
rating_status_disagreements,0
countries,15


,restaurant_id,country,aggregate_rating,rating_for_analysis,cuisines_missing
0,53,India,2.6,2.6,False
1,55,India,3.9,3.9,False
2,60,India,3.2,3.2,False
3,64,India,3.8,3.8,False
4,65,India,3.5,3.5,False


## Build saved outputs

This runs the same code used by the command-line workflow. Exact input hashes and audit counts are recorded in reports/data_quality.json. No statistical imputation or outlier removal is applied.

In [4]:
_, results = run_pipeline()
display(cuisines.head())

{
  "raw_rows": 9551,
  "clean_rows": 9551,
  "exact_duplicates_removed": 0,
  "duplicate_restaurant_ids": 0,
  "missing_cuisines": 9,
  "unrated_restaurants": 2148,
  "zero_cost_records": 18,
  "zero_coordinate_pairs": 497,
  "rating_status_disagreements": 0,
  "countries": 15,
  "country_city_pairs": 141,
  "source_sha256": {
    "zomato.csv": "100f5b85699357666b8e8e7ae57d18c723c47fe2a6b47f59809cbbe9d19d21b2",
    "Country-Code.xlsx": "c8f2f80c8d54122563037c4ee20593b42fe21f95c19d055fc5e4d6f4c67dba0a"
  }
}


,restaurant_id,cuisine
0,53,Chinese
1,53,Mughlai
2,53,North Indian
3,55,Chinese
4,55,Thai


## Data limitations

Missing cuisines remain null. Latin-1 decoding makes the original file readable but cannot repair already corrupted names. Country labels come from the supplied lookup. Currency values are preserved despite apparent inconsistencies; no global monetary comparison is valid.